# Ingestão diária de notícias — Daily Infra

Este notebook é um **exemplo didático** de como fazer ingestão de documentos
(notícias, no caso) num **Volume do Unity Catalog** para uso posterior por
pipelines de LLM / RAG.

Passos executados:
1. Para cada query, consulta o **RSS do Google News** filtrando as últimas
   24 horas (`when:1d`).
2. Extrai os itens (título, link redirecionado do Google, data, resumo).
3. **Decodifica** o link do Google News para chegar na URL real da matéria
   (usando `googlenewsdecoder`).
4. Faz o download do HTML da matéria usando **técnicas anti-bot** leves:
   - `curl_cffi` (impersonando um Chrome real, com fingerprint TLS/JA3),
   - rotação de `User-Agent` e headers,
   - jitter aleatório entre requisições,
   - fallback para `httpx` puro caso `curl_cffi` falhe.
5. **Limpa** o HTML e extrai só o texto útil.
6. Salva no Volume dois arquivos por matéria:
   - `{source}_{article_name}.txt`  → texto limpo
   - `{source}_{article_name}.json` → metadados (título, url, data, query, etc.)

In [0]:
%pip install --quiet feedparser beautifulsoup4 googlenewsdecoder curl_cffi httpx lxml
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
from datetime import datetime, timezone
from typing import Optional

import feedparser
import httpx
from bs4 import BeautifulSoup
from email.utils import parsedate_to_datetime
from googlenewsdecoder import gnewsdecoder

# curl_cffi permite "impersonar" um navegador real no nível TLS/JA3.
# Isso ajuda muito a passar por sites que fazem bot-detection agressivo
# (Cloudflare, DataDome etc.), porque o fingerprint da conexão parece
# um Chrome de verdade e não uma lib Python.
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

# Data de referência (usada no nome da pasta destino).
HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

# Pasta destino no Volume do Unity Catalog.
# Volumes do UC são montados no filesystem local do driver em /Volumes/...,
# então podemos usar `open()` normal do Python — não precisa de dbutils.fs.
PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}GERAL"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"[setup] Salvando artefatos em: {PASTA_DESTINO}")

# Lista de queries do daily Infra. Cada item é a string que vai no `q=` do
# Google News. Já usamos as aspas e OR do jeito que o Google espera.
QUERIES = ["CENTRAL FOTOVOLTAICA SãO PEDRO IV S.A.",
            "SPIV11",
            "São Pedro IV S.A.",
            "Centrais Fotovoltaicas São Pedro II e IV",
            "CENTRAL FOTOVOLTAICA SãO PEDRO IV S.A.",
            "São Pedro IV",
            "São Pedro",
            "OURINHOS SANEAMENTO S.A.",
            "OURO PRETO SERVICOS DE SANEAMENTO S.A - SANEOURO",
            "OUSA11",
            "Ourinhos Saneamento",
            "Consorcio GS Inima ã Traãado",
            "Consorcio GS INIMA ã TRAãADO",
            "Traãado de Erechim",
            "GS INIMA ã Traãado",
            "GS Inima Brasil",
            "OURINHOS SANEAMENTO S.A.",
            "Saneamento Ourinhos",
            "SOBRAL I SOLAR ENERGIA SPE S.A.",
            "SISE11",
            "Sobral Solar Energia",
            "Sobral I Solar",
            "Sobral Energia",
            "SOBRAL I SOLAR ENERGIA SPE S.A.",
            "TRANSMISSORA ALIANCA DE ENERGIA ELETRICA S/A",
            "TRANSMISSORA ALIANCA DE ENERGIA ELETRICA S/A",
            "TAEE18",
            "Transmissora Alianãa de Energia Elãtrica",
            "TAEE4",
            "Transmissora Alianãa de Energia Elãtrica S/A",
            "TAEE3",
            "TAEE11",
            "TAEE3F",
            "TAEE4F",
            "Transmissora Alianca de Energia Eletrica S.A.",
            "TAESA",
            "Taesa",
            "TAEE11F",
            "COMERC Participações S.A.",
            "HãLIO VALGAS SOLAR Participações S.A.",
            "COMR14",
            "Vibra",
            "Vibra Energia",
            "VBBR3",
            "Comerc",
            "Vibra Energia S.A",
            "Comerc Energia",
            "BR Distribuidora",
            "VENTOS DE SAO CLEMENTE HOLDING S.A",
            "VENTOS DE SAO CLEMENTE HOLDING S.A",
            "VSCL12",
            "Complexo Eólico Ventos de São Clemente",
            "São Clemente",
            "Ventos de São Clemente",
            "Echoenergia",
            "Complexo Ventos de São Clemente",
            "VENTOS DE SAO CLEMENTE HOLDING S.A",
            "RIO + SANEAMENTO BL3 S.A",
            "SABP12",
            "RIO + SANEAMENTO BL3 S.A",
            "Rio+ Saneamento BL3 S.A",
            "Rio+Saneamento",
            "Rio+ Saneamento",
            "RUMO MALHA PAULISTA S/A",
            "GASC18",
            "Sulgãs",
            "Compass",
            "Cosan",
            "Rumo",
            "Radar",
            "OneBio",
            "Orizon",
            "Edge",
            "Raãzen",
            "SCGãS",
            "CEG-Rio",
            "MSGãS",
            "Cosan Nove Participações",
            "Moove",
            "Commit",
            "Comgãs",
            "CSAN3",
            "Compagas",
            "PORTOCEM GERAÇãO DE ENERGIA S.A.",
            "PTCE11",
            "New Fortress Energy",
            "Portocem",
            "UTE Portocem",
            "NFE",
            "Portocem GERAÇãO de Energia S.A.",
            "Ceiba Energy",
            "UTE Portocem I",
            "RZK SOLAR 07 S.A.",
            "RZKS11",
            "Grupo RZK",
            "RZK Energia S.A.",
            "RZK Soluções e Participações",
            "RZK Solar 02",
            "Thopen",
            "RZK Comercializadora",
            "RZK Energia",
            "Nova Milano",
            "RZK Soluções",
            "RZK",
            "TROPICALIA TRANSMISSORA DE ENERGIA S.A.",
            "TRPI13",
            "TROPICALIA TRANSMISSORA DE ENERGIA S.A.",
            "Holbaeck Empreendimentos e Participações",
            "Tropicãlia Transmissora de Energia",
            "Tropicãlia",
            "HIGHLINE DO BRASIL II INFRAESTRUTURA DE TELECOMUNICACOES S.A.",
            "HIGHLINE DO BRASIL II INFRAESTRUTURA DE TELECOMUNICACOES S.A.",
            "HGLB23",
            "Highline do Brasil",
            "Highline",
            "Highline do Brasil II Infraestrutura de Telecomunicaãães S.A.",
            "HIGHLINE DO BRASIL II INFRAESTRUTURA DE TELECOMUNICACOES S.A.",
            "Highline do Brasil II",
            "DigitalBridge",
            "Digital Colony",
            "Highline Brasil",
            "USINA DE ENERGIA FOTOVOLTAICA DE COROMANDEL S.A.",
            "CRMD11",
            "Usina Fotovoltaica de Coromandel",
            "Solatio",
            "Usina de Energia Fotovoltaica de Coromandel S.A.",
            "Coromandel 2",
            "UFV Coromandel",
            "AGUAS DO RIO 4 SPE S.A",
            "AGUAS DO RIO 1 SPE S.A",
            "AGUAS DO RIO 4 SPE S.A",
            "AGUAS DO RIO 1 SPE S.A",
            "AGUAS DO RIO 1 SPE S.A",
            "AEGEA SANEAMENTO E PARTICIPACOES S.A.",
            "AGUAS DO RIO 4 SPE S.A",
            "RIS424",
            "AEGEA SANEAMENTO E PARTICIPACOES S.A.",
            "Aegea Saneamento",
            "Aegea",
            "BC GERACAO E COMERCIALIZACAO DE ENERGIA S/A",
            "BCGE12",
            "BC GERAÇãO",
            "BC Energia",
            "BC GERACAO E COMERCIALIZACAO DE ENERGIA S/A",
            "TOTALENERGIES DRACENA PARTICIPACOES S.A.",
            "TOTALENERGIES DRACENA PARTICIPACOES S.A.",
            "EREN23",
            "Total Energies",
            "TotalEnergies",
            "TOTALENERGIES",
            "TOTALENERGIES DRACENA PARTICIPACOES S.A.",
            "CENTRAL FOTOVOLTAICA SãO PEDRO II S.A.",
            "FTSP11",
            "CENTRAL FOTOVOLTAICA SãO PEDRO II S.A.",
            "UTE GNA I GERAÇãO DE ENERGIA S.A.",
            "UNEG11",
            "UTE GNA I",
            "UTE GNA",
            "UTE GNA I GERAÇãO DE ENERGIA S.A.",
            "GNA I",
            "CONCESSIONãRIO VIA RIO S.A.",
            "CONCESSIONãRIO VIA RIO S.A.",
            "CTOL18",
            "CCR S.A.",
            "CCR",
            "COPACABANA GERAÇãO DE ENERGIA E Participações S.A.",
            "CGEP12",
            "COPACABANA GERAÇãO DE ENERGIA E Participações S.A.",
            "Copacabana GERAÇãO",
            "Copacabana Energia S.A.",
            "Copacabana Energia",
            "PRS Aeroportos SA",
            "PAX INVESTIMENTOS EM AEROPORTOS S.A",
            "PRSS11",
            "PAX Investimentos",
            "PAX Investimentos em Aeroportos S.A",
            "PAX Aeroportos",
            "PAX",
            "TRANSMISSORA JOSE MARIA DE MACEDO DE ELETRICIDADE S.A.",
            "TJMM11",
            "TRANSMISSORA JOSE MARIA DE MACEDO DE ELETRICIDADE S.A.",
            "GUARACIABA TRANSMISSORA DE ENERGIA (TP SUL) S.A.",
            "TPSU12",
            "GUARACIABA TRANSMISSORA",
            "TP SUL",
            "GUARACIABA TRANSMISSORA DE ENERGIA",
            "TP SUL S.A.",
            "ROTA DO PARã S.A.",
            "RPAS11",
            "ROTA DO PARã S.A.",
            "Rota do Parã",
            "SPE TRANSMISSORA DE ENERGIA LINHA VERDE II S.A.",
            "SPLV11",
            "Linha Verde II",
            "SPE Transmissora Linha Verde II",
            "SPE TRANSMISSORA DE ENERGIA LINHA VERDE II S.A.",
            "AXS ENERGIA S.A.",
            "AXSD11",
            "AXS ENERGIA S.A.",
            "AXS",
            "AXS Energia",
            "VINEYARDS TRANSMISSãO DE ENERGIA",
            "VNYD12",
            "VINEYARDS TRANSMISSãO DE ENERGIA",
            "V2I ENERGIA S.A.",
            "V2I ENERGIA S.A.",
            "VDIE12",
            "V2I ENERGIA S.A.",
            "V2I",
            "V2I Energia",
            "VENTOS DE SANTO ESTEVAO HOLDING S/A",
            "VSEH11",
            "AUREN ENERGIA S.A.",
            "Auren",
            "Auren Energia",
            "Auren Energia S.A.",
            "PAMPA TRANSMISSãO DE ENERGIA S.A.",
            "PAMP12",
            "PAMPA TRANSMISSãO DE ENERGIA S.A.",
            "Pampa Energia",
            "Pampa Transmissão",
            "ARCOVERDE TRANSMISSAO DE ENERGIA S.A.",
            "ARCV12",
            "Arcoverde Energia",
            "ARCOVERDE TRANSMISSAO DE ENERGIA S.A.",
            "Arcoverde Transmissão",
            "Arcoverde",
            "VENTOS DE SAO JOAO XXIII ENERGIAS RENOVAVEIS S.A.",
            "VSJX11",
            "VENTOS DE SAO JOAO XXIII ENERGIAS RENOVAVEIS S.A.",
            "VENTOS DE SAO LUCIO I ENERGIAS RENOVAVEIS S.A.",
            "VSLE11",
            "VENTOS DE SAO LUCIO I ENERGIAS RENOVAVEIS S.A.",
            "IGUA RIO DE JANEIRO S.A",
            "IGUA RIO DE JANEIRO S.A",
            "IRJS15",
            "IGUA SANEAMENTO S.A.",
            "Igua Saneamento",
            "Igua",
            "CONCESSIONÁRIA PONTE RIO-NITERÓI S.A. (ECOPONTE)",
            "ECPN11",
            "CONCESSIONÁRIA PONTE RIO-NITERÓI S.A.",
            "Ponte Rio-Niterói",
            "ECOPONTE",
            "Ecoponte",
            "CONCESSIONARIA RODOVIAS DO SUL DE MINAS SPE S.A.",
            "RDSM12",
            "Rodovias Sul de Minas SPE",
            "Concessionãria Rodovias do Sul",
            "Rodovias do Sul de Minas",
            "CONCESSIONARIA RODOVIAS DO SUL DE MINAS SPE S.A.",
            "CONCESSIONãRIA ROTA DOS COQUEIROS S.A.",
            "RTCQ12",
            "Monte Capital Management",
            "Monte Capital",
            "Monte Capital Management Gestora",
            "MONTE CAPITAL MANAGEMENT GESTORA DE RECURSOS S.A.",
            "GDPAR SN Participações em Projetos Solares S.A.",
            "GDPAR SR PARTICIPACOES EM PROJETOS SOLARES S.A.",
            "RBRA19",
            "GD - GERACAO DISTRIBUIDA PARTICIPACOES S.A",
            "GD Geração Distribuida",
            "GD Participações",
            "CONCESSIONARIA DAS RODOVIAS AYRTON SENNA E CARVALHO PINTO S/A - ECOPISTAS",
            "ASCP23",
            "Ecopistas S.A.",
            "CONCESSIONARIA DAS RODOVIAS AYRTON SENNA E CARVALHO PINTO S/A - ECOPISTAS",
            "Ecopistas",
            "Concessionãria Ecopistas",
            "VILA PIAUI 2 EMPREENDIMENTOS E PARTICIPACOES S.A.",
            "VILA PIAUI 1 EMPREENDIMENTOS E PARTICIPACOES S.A.",
            "VP2E11",
            "Vila Piauã",
            "VILA PIAUI 2",
            "Vila Piauã 2",
            "VILA PIAUI 2 EMPREENDIMENTOS E PARTICIPACOES S.A.",
            "ãGUAS DO SERTãO S.A.",
            "ASER12",
            "CONASA",
            "Conasa Infraestrutura",
            "CONASA INFRAESTRUTURA S.A.",
            "JANUARIO DE NAPOLI GERACAO DE ENERGIA S/A",
            "JANU11",
            "JANUARIO DE NAPOLI GERACAO DE ENERGIA S/A",
            "ECORODOVIAS CONCESSãES E SERVIãOS S.A. (SUBHOLDING)",
            "ECO135 CONCESSIONARIA DE RODOVIAS S.A.",
            "ERDVB4",
            "Ecorodovias Subholding",
            "Ecorodovias",
            "Ecorodovias Concessães",
            "ECORODOVIAS CONCESSãES E SERVIãOS S.A.",
            "CALDEIRAO GRANDE ENERGIAS RENOVAVEIS S/A",
            "CLGE11",
            "CALDEIRAO GRANDE ENERGIAS RENOVAVEIS S/A",
            "SOLARIS TRANSMISSAO DE ENERGIA S.A",
            "SOTE11",
            "Solaris Transmissão de Energia S.A",
            "Solaris Transmissão",
            "Solaris Energia",
            "BRK AMBIENTAL PARTICIPACOES S.A.",
            "BRK AMBIENTAL PARTICIPACOES S.A.",
            "BRK AMBIENTAL URUGUAIANA S.A.",
            "BRK AMBIENTAL - REGIAO METROPOLITANA DE MACEIO S.A.",
            "BRK AMBIENTAL - MAUA S.A.",
            "BRKPA1",
            "BRK",
            "BRK Ambiental Participacoes S.A.",
            "BRK Ambiental S.A.",
            "BRK Ambiental",
            "CONCESSIONARIA VIARIO S",
            "CTOL28",
            "CTOL18"]

# Pool de User-Agents "reais" para rotacionar. Uma medida simples porém eficaz.
USER_AGENTS = [
    # Chrome desktop
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    # Firefox desktop
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
    # Edge desktop
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0",
    # Safari macOS
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
]

# Perfis de "impersonation" que a curl_cffi conhece.
# A cada request escolhemos um aleatório para variar o fingerprint TLS.
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124", "safari17_0", "edge101"]

# Timeout padrão pras requisições HTTP.
HTTP_TIMEOUT = 30

In [0]:
# =============================================================================
# Helpers
# =============================================================================

def slugify(texto: str, max_len: int = 80) -> str:
    """
    Transforma uma string qualquer em um "slug" seguro pra ser usado
    como nome de arquivo (sem acento, sem espaço, sem caractere estranho).
    """
    if not texto:
        return "sem-titulo"
    # Remove acentos.
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    # Troca qualquer coisa que não seja [a-zA-Z0-9] por hífen.
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    # Corta pra não estourar limite de nome de arquivo.
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    """Hash MD5 curto — útil pra desambiguar nomes de arquivo iguais."""
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def parsear_data_rss(data_str: str) -> str:
    """
    Converte uma string de data no formato RFC 2822 do RSS
    (ex.: 'Sun, 22 Jun 2026 12:00:00 GMT') para 'YYYY-MM-DD'.
    Devolve a data de hoje como fallback se o parse falhar.
    """
    try:
        return parsedate_to_datetime(data_str).strftime("%Y-%m-%d")
    except Exception:
        return HOJE


def domain_from_url(url: str) -> str:
    """Extrai o domínio (source) de uma URL. Ex.: 'valor.globo.com'."""
    try:
        netloc = urllib.parse.urlparse(url).netloc.lower()
        # Remove 'www.' pra ficar mais limpo.
        return netloc[4:] if netloc.startswith("www.") else netloc
    except Exception:
        return "desconhecido"


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    """Monta um dicionário de headers HTTP parecido com o de um browser real."""
    ua = random.choice(USER_AGENTS)
    headers = {
        "User-Agent": ua,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,"
                  "image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1",
    }
    if referer:
        headers["Referer"] = referer
    return headers


In [0]:
# =============================================================================
# Etapa 1 — Consultar o RSS do Google News
# =============================================================================

def buscar_google_news(query: str, janela: str = "1d") -> list[dict]:
    """
    Consulta o RSS de busca do Google News para uma query, restringindo
    à janela temporal informada (default: 1 dia).

    Retorna lista de dicts com: title, link, published, summary.
    """
    base_url = "https://news.google.com/rss/search"
    # `when:1d` restringe pras últimas 24h. É colado dentro do próprio q.
    params = {
        "q": f"{query} when:{janela}",
        "hl": "pt-BR",
        "gl": "BR",
        "ceid": "BR:pt-419",
    }
    full_url = f"{base_url}?{urllib.parse.urlencode(params)}"

    feed = feedparser.parse(full_url)

    itens = []
    for entry in feed.entries:
        # O 'summary' vem em HTML — a gente limpa pra texto puro.
        summary_txt = ""
        if getattr(entry, "summary", None):
            summary_txt = BeautifulSoup(entry.summary, "lxml").get_text(" ", strip=True)

        itens.append({
            "title": getattr(entry, "title", ""),
            "link": getattr(entry, "link", ""),
            "published": getattr(entry, "published", ""),
            "summary": summary_txt,
            # Google News manda a fonte dentro de 'source' em alguns casos.
            "source_google": getattr(getattr(entry, "source", None), "title", "") or "",
        })
    return itens


In [0]:
# =============================================================================
# Etapa 2 — Decodificar o link do Google News para a URL real
# =============================================================================

def decodificar_link_google(link_google: str) -> Optional[str]:
    """
    Google News não devolve a URL final da matéria no RSS; ele devolve um
    link intermediário do domínio news.google.com. Essa função usa a lib
    `googlenewsdecoder` pra chegar na URL real.
    """
    try:
        r = gnewsdecoder(link_google)
        if r.get("status") is True:
            return r.get("decoded_url")
    except Exception as e:
        print(f"  [decoder] falhou: {e}")
    return None


In [0]:
# =============================================================================
# Etapa 3 — Baixar o HTML com técnicas anti-bot
# =============================================================================

def baixar_html(url: str, tentativas: int = 3) -> Optional[str]:
    """
    Tenta baixar o HTML da URL usando:
      1) curl_cffi com fingerprint de Chrome real (passa a maioria dos WAFs).
      2) Fallback pra httpx puro caso o curl_cffi devolva erro.

    Faz até `tentativas` tentativas com backoff aleatório.
    """
    for i in range(1, tentativas + 1):
        # Sleep aleatório entre requisições — evita cadência robótica.
        time.sleep(random.uniform(0.8, 2.2))

        headers = headers_aleatorios(referer="https://news.google.com/")
        impersonate = random.choice(IMPERSONATE_PROFILES)

        # --- Tentativa via curl_cffi ---
        try:
            resp = cffi_requests.get(
                url,
                headers=headers,
                impersonate=impersonate,
                timeout=HTTP_TIMEOUT,
                allow_redirects=True,
            )
            # 2xx = ok. Alguns sites devolvem 403 pra bot; tentamos de novo.
            if 200 <= resp.status_code < 300 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {i}] status={resp.status_code} "
                  f"len={len(resp.text) if resp.text else 0}")
        except Exception as e:
            print(f"  [curl_cffi tent {i}] erro: {e}")

        # --- Fallback via httpx ---
        try:
            with httpx.Client(
                headers=headers,
                follow_redirects=True,
                timeout=HTTP_TIMEOUT,
                http2=True,
            ) as client:
                resp = client.get(url)
                if 200 <= resp.status_code < 300 and resp.text and len(resp.text) > 500:
                    return resp.text
                print(f"  [httpx tent {i}] status={resp.status_code} "
                      f"len={len(resp.text) if resp.text else 0}")
        except Exception as e:
            print(f"  [httpx tent {i}] erro: {e}")

    return None


In [0]:
# =============================================================================
# Etapa 4 — Limpar o HTML e extrair só o texto útil
# =============================================================================

# Tags cujo conteúdo raramente serve pra análise textual.
TAGS_LIXO = [
    "script", "style", "noscript", "iframe", "svg", "form",
    "nav", "footer", "header", "aside", "button",
]

def extrair_texto(html: str) -> str:
    """
    Recebe o HTML bruto de uma matéria e devolve o texto "limpo":
    - remove tags de navegação/estilo/script,
    - prioriza o conteúdo dentro de <article> quando existe,
    - colapsa espaços em branco.
    """
    if not html:
        return ""

    soup = BeautifulSoup(html, "lxml")

    # Remove tags de lixo.
    for tag in soup(TAGS_LIXO):
        tag.decompose()

    # Se o site marcou <article>, usamos só isso (bem mais limpo).
    article = soup.find("article")
    base = article if article else soup

    texto = base.get_text("\n", strip=True)
    # Colapsa múltiplas quebras de linha.
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


In [0]:
# =============================================================================
# Etapa 5 — Salvar no Volume
# =============================================================================

def salvar_artefatos(
    pasta: str,
    source: str,
    titulo: str,
    texto: str,
    metadados: dict,
) -> tuple[str, str]:
    """
    Persiste no Volume os dois arquivos por matéria:
      - {source}_{article_name}.txt  (texto limpo)
      - {source}_{article_name}.json (metadados)

    O nome usa slug do título + hash curto do link, pra:
      (a) ficar legível pra humanos,
      (b) não colidir quando dois títulos batem.
    """
    slug_source = slugify(source, max_len=40) or "fonte"
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url_final") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    # Escreve o TXT com o corpo da matéria.
    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")

    # Escreve o JSON com os metadados.
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


In [0]:
# =============================================================================
# Etapa 6 — Pipeline principal
# =============================================================================

def processar_query(query: str, pasta_destino: str) -> list[dict]:
    """
    Executa o pipeline inteiro para uma query:
    RSS -> decode -> download -> limpeza -> save.
    Devolve uma lista com o resumo do que foi feito (pra log).
    """
    print(f"\n=== Query: {query!r} ===")
    resultados = []

    itens = buscar_google_news(query, janela="1d")
    print(f"  {len(itens)} itens no RSS.")

    for i, item in enumerate(itens, start=1):
        print(f"\n  [{i}/{len(itens)}] {item['title'][:100]}")

        # 1) Decodifica o link do Google News.
        url_final = decodificar_link_google(item["link"])
        if not url_final:
            print("    -> não consegui decodificar o link; pulando.")
            continue

        # 2) Baixa o HTML da matéria.
        html = baixar_html(url_final)
        if not html:
            print("    -> download do HTML falhou; pulando.")
            continue

        # 3) Extrai texto.
        texto = extrair_texto(html)
        if not texto or len(texto) < 200:
            print(f"    -> texto muito curto ({len(texto)} chars); pulando.")
            continue

        # 4) Monta metadados no padrão canônico de ingestão:
        #    source_id / title / description / url / date / published_at
        source = domain_from_url(url_final) or item.get("source_google") or "desconhecido"
        data_publicacao = parsear_data_rss(item["published"])
        metadados = {
            "source_id": "linked_article",
            "title": item["title"],
            # description segue o padrão "Linked from <origem>" — equivalente
            # ao "Linked from email AAUHF0WWAAA=" do padrão de referência.
            "description": f"Linked from Google News query: {query}",
            "url": url_final,
            # date = data da coleta (dia de hoje)
            "date": HOJE,
            # published_at = data de publicação reportada pelo RSS
            "published_at": data_publicacao,
        }

        # 5) Salva no Volume.
        caminho_txt, caminho_json = salvar_artefatos(
            pasta=pasta_destino,
            source=source,
            titulo=item["title"],
            texto=texto,
            metadados=metadados,
        )
        print(f"    -> salvo em {caminho_txt}")

        resultados.append({
            "query": query,
            "titulo": item["title"],
            "source": source,
            "url_final": url_final,
            "caminho_txt": caminho_txt,
            "caminho_json": caminho_json,
        })

    return resultados


In [0]:
# =============================================================================
# Execução
# =============================================================================

todos_resultados: list[dict] = []
for q in QUERIES:
    try:
        todos_resultados.extend(processar_query(q, PASTA_DESTINO))
    except Exception as e:
        # Uma query quebrada não pode derrubar o job inteiro.
        print(f"[ERRO] query {q!r} falhou: {e}")

print(f"\n\n=== Fim. {len(todos_resultados)} matérias salvas em {PASTA_DESTINO} ===")